# Optimización de Carteras con Datos Reales

**Universidad Internacional de La Rioja (UNIR)**  
**Profesor:** Dr. Rodolfo Rafael Medina Ramírez

---

## 🎯 Objetivos

1. Descargar datos reales con yfinance
2. Optimizar carteras
3. Realizar backtesting
4. Comparar con S&P 500

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from datetime import datetime, timedelta
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

print('✅ Bibliotecas importadas')

In [ ]:
# Configurar activos y periodo
tickers = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA', 'JPM', 'JNJ', 'XOM', 'WMT', 'PG']
fecha_fin = datetime.now()
fecha_inicio = fecha_fin - timedelta(days=365*3)

print(f'Descargando {len(tickers)} activos...')

In [ ]:
# Descargar datos
try:
    datos = yf.download(tickers, start=fecha_inicio, end=fecha_fin, progress=False)
    precios = datos['Adj Close'].dropna()
    print(f'✅ Descargados: {len(precios)} días')
except Exception as e:
    print(f'Error: {e}')
    print('Usando datos de ejemplo...')
    precios = pd.read_csv('../recursos/datasets/precios_historicos_completo.csv', 
                          index_col='fecha', parse_dates=True)

In [ ]:
# Calcular rendimientos
rendimientos = np.log(precios / precios.shift(1)).dropna()
print('Primeros rendimientos:')
print(rendimientos.head())

In [ ]:
# Parámetros anualizados
dias_trading = 252
rendimientos_anuales = rendimientos.mean() * dias_trading
volatilidades_anuales = rendimientos.std() * np.sqrt(dias_trading)
matriz_cov = rendimientos.cov() * dias_trading

print('Estadísticas calculadas')

## Clase Optimizadora

In [ ]:
class OptimizadorCarteras:
    
    def __init__(self, rendimientos_esp, cov_matrix, rf=0.03):
        self.rendimientos = np.array(rendimientos_esp)
        self.cov_matrix = np.array(cov_matrix)
        self.rf = rf
        self.n_activos = len(rendimientos_esp)
    
    def rendimiento_cartera(self, pesos):
        return np.dot(pesos, self.rendimientos)
    
    def volatilidad_cartera(self, pesos):
        return np.sqrt(np.dot(pesos.T, np.dot(self.cov_matrix, pesos)))
    
    def sharpe_ratio(self, pesos):
        ret = self.rendimiento_cartera(pesos)
        vol = self.volatilidad_cartera(pesos)
        return (ret - self.rf) / vol
    
    def optimizar_tangente(self):
        def objetivo(w):
            return -self.sharpe_ratio(w)
        
        restricciones = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
        limites = tuple((0, 1) for _ in range(self.n_activos))
        pesos_init = np.array([1/self.n_activos] * self.n_activos)
        
        resultado = minimize(objetivo, pesos_init, method='SLSQP',
                           bounds=limites, constraints=restricciones)
        
        pesos = resultado.x
        
        return {
            'pesos': pesos,
            'rendimiento': self.rendimiento_cartera(pesos),
            'volatilidad': self.volatilidad_cartera(pesos),
            'sharpe': self.sharpe_ratio(pesos)
        }

print('✅ Clase definida')

In [ ]:
# Optimizar
optimizador = OptimizadorCarteras(rendimientos_anuales.values, matriz_cov.values, rf=0.03)
tangente = optimizador.optimizar_tangente()

print('⭐ CARTERA TANGENTE')
print(f"Rendimiento: {tangente['rendimiento']*100:.2f}%")
print(f"Volatilidad: {tangente['volatilidad']*100:.2f}%")
print(f"Sharpe: {tangente['sharpe']:.4f}")
print('\nComposición:')
for i, ticker in enumerate(tickers):
    if tangente['pesos'][i] > 0.01:
        print(f"  {ticker}: {tangente['pesos'][i]*100:.2f}%")

---

**UNIR - Agosto 2025**  
*Para dudas: Foro Moodle*